# ALPR Training Pipeline (Colab)

This notebook trains and evaluates both models:
- Plate detection model (YOLOv8)
- Plate character recognition model (YOLOv8)

It stores all outputs under `artifacts/`:
- `artifacts/detection/`
- `artifacts/recognition/`
- `artifacts/models/`
- `artifacts/eval_images/`

Run cells from top to bottom in Colab with GPU enabled.

In [26]:
# 1) Install dependencies (Colab)
# If you run locally and already installed deps, this is safe to keep.
%pip install -q -U pip
%pip install -q ultralytics opencv-python pyyaml onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.8 MB/s eta 0:00:0000:0100:01


In [41]:
# 2) Resolve project directory and create artifacts folders
from pathlib import Path
import os
import shutil
import subprocess
import sys

POSSIBLE_PATHS = [  # common Colab clone path
    Path('/content/../traffic surveillance system/ai-service'),
    Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
    Path('/run/media/kareem_taha/01D8ACF78D3044C0/college/computer science level 4/second term/smart applications/traffic surveillance system/ai-service'),
]

PROJECT_DIR = None
for p in POSSIBLE_PATHS:
    if (p / 'scripts').exists() and (p / 'configs').exists():
        PROJECT_DIR = p
        break

if PROJECT_DIR is None:
    raise RuntimeError('Could not locate ai-service project directory. Set PROJECT_DIR manually.')

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)

ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
DETECTION_ARTIFACTS = ARTIFACTS_DIR / 'detection'
RECOGNITION_ARTIFACTS = ARTIFACTS_DIR / 'recognition'
MODELS_ARTIFACTS = ARTIFACTS_DIR / 'models'
EVAL_IMAGES_DIR = ARTIFACTS_DIR / 'eval_images'

for d in [ARTIFACTS_DIR, DETECTION_ARTIFACTS, RECOGNITION_ARTIFACTS, MODELS_ARTIFACTS, EVAL_IMAGES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Artifacts folder ready:', ARTIFACTS_DIR)

RuntimeError: Could not locate ai-service project directory. Set PROJECT_DIR manually in this cell.

In [ ]:
# 3) Get dataset (clone if missing)
DATASET_DIR = PROJECT_DIR / 'dataset'
if not DATASET_DIR.exists():
    subprocess.run([
        'git', 'clone', 'https://github.com/ahmedramadan96/EALPR.git', str(DATASET_DIR)
    ], check=True)
else:
    print('Dataset already exists:', DATASET_DIR)

# quick sanity check
for rel in [
    'EALPR Vechicles dataset/Vehicles',
    'EALPR Vechicles dataset/Vehicles Labeling',
    'EALPR- Plates dataset',
    'EALPR- LP characters dataset/Characters Labeling',
]:
    p = DATASET_DIR / rel
    print(rel, 'OK' if p.exists() else 'MISSING')

In [ ]:
# 4) Prepare detection and recognition datasets
subprocess.run([
    sys.executable, 'scripts/prepare_ealpr.py',
    '--raw', 'dataset',
    '--out', 'data/processed/detection'
], check=True)

subprocess.run([
    sys.executable, 'scripts/prepare_ealpr_recognition.py',
    '--raw', 'dataset',
    '--out', 'data/processed/recognition'
], check=True)

print('Dataset preparation complete.')

In [ ]:
# 5) Train detection model (YOLOv8)
from ultralytics import YOLO

det_data_yaml = PROJECT_DIR / 'data/processed/detection/data.yaml'
det_model = YOLO('yolov8n.pt')

det_model.train(
    data=str(det_data_yaml),
    epochs=50,
    imgsz=640,
    device=0,  # use GPU in Colab
    project=str(DETECTION_ARTIFACTS),
    name='train',
    exist_ok=True,
)

print('Detection training complete.')

In [ ]:
# 6) Evaluate detection model and save prediction images
import shutil

det_weights = DETECTION_ARTIFACTS / 'train' / 'weights' / 'best.pt'
assert det_weights.exists(), f'Detection best weights not found: {det_weights}'

det_best = YOLO(str(det_weights))
det_metrics = det_best.val(data=str(det_data_yaml), split='val', device=0)
print('Detection mAP50:', getattr(getattr(det_metrics, 'box', None), 'map50', None))

det_val_images = PROJECT_DIR / 'data/processed/detection/splits/images/val'
det_eval_out = EVAL_IMAGES_DIR / 'detection'
det_eval_out.mkdir(parents=True, exist_ok=True)

det_best.predict(
    source=str(det_val_images),
    device=0,
    save=True,
    conf=0.25,
    project=str(det_eval_out),
    name='preds',
    exist_ok=True,
)

print('Detection evaluation images saved to:', det_eval_out)

In [ ]:
# 7) Train recognition model (YOLOv8 char detector)
rec_data_yaml = PROJECT_DIR / 'data/processed/recognition/data.yaml'
rec_model = YOLO('yolov8n.pt')

rec_model.train(
    data=str(rec_data_yaml),
    epochs=80,
    imgsz=320,
    device=0,
    project=str(RECOGNITION_ARTIFACTS),
    name='train',
    exist_ok=True,
)

print('Recognition training complete.')

In [ ]:
# 8) Evaluate recognition model and save prediction images
rec_weights = RECOGNITION_ARTIFACTS / 'train' / 'weights' / 'best.pt'
assert rec_weights.exists(), f'Recognition best weights not found: {rec_weights}'

rec_best = YOLO(str(rec_weights))
rec_metrics = rec_best.val(data=str(rec_data_yaml), split='val', device=0)
print('Recognition mAP50:', getattr(getattr(rec_metrics, 'box', None), 'map50', None))

rec_val_images = PROJECT_DIR / 'data/processed/recognition/splits/images/val'
rec_eval_out = EVAL_IMAGES_DIR / 'recognition'
rec_eval_out.mkdir(parents=True, exist_ok=True)

rec_best.predict(
    source=str(rec_val_images),
    device=0,
    save=True,
    conf=0.25,
    project=str(rec_eval_out),
    name='preds',
    exist_ok=True,
)

print('Recognition evaluation images saved to:', rec_eval_out)

In [ ]:
# 9) Copy best models to artifacts/models and summarize outputs
final_det = MODELS_ARTIFACTS / 'plate-detector.pt'
final_rec = MODELS_ARTIFACTS / 'plate-characters.pt'

shutil.copy2(det_weights, final_det)
shutil.copy2(rec_weights, final_rec)

print('Saved detector model:', final_det)
print('Saved recognizer model:', final_rec)
print('Artifacts root:', ARTIFACTS_DIR)

for p in [DETECTION_ARTIFACTS, RECOGNITION_ARTIFACTS, EVAL_IMAGES_DIR, MODELS_ARTIFACTS]:
    print('-', p)

In [ ]:
# 10) Optional: zip artifacts for easy download from Colab
archive_path = PROJECT_DIR / 'artifacts.zip'
if archive_path.exists():
    archive_path.unlink()

shutil.make_archive(str(archive_path.with_suffix('')), 'zip', root_dir=ARTIFACTS_DIR)
print('Created:', archive_path)

# In Colab you can download with:
# from google.colab import files
# files.download(str(archive_path))